# Scientific assessment of nsb-py: observational performance and comparison with nsb2

This notebook assesses **nsb-py** against measured night-sky spectra and nsb2.
It presents prediction errors, uncertainty and model assumptions for astronomers,
observers and scientific reviewers.

NSB predicts photon radiance for an observing location, time and sky direction by
combining **airglow, zodiacal light, integrated starlight and scattered moonlight**.
Here we compare its predictions with flux-calibrated **Patat/FORS1 observations
at Paranal**, and with the independent implementation **nsb2**, over **365–650 nm**.
The observable is radiance in photons cm⁻² ns⁻¹ sr⁻¹, rather than detector counts
or a photometric magnitude. The selected observations are dark-time and moonless;
they do not test the moonlight model or NSB's entire default 300–650 nm band.

The analysis addresses three questions:

1. **Observational agreement:** how large are the prediction errors and systematic biases?
2. **Comparative performance:** how does NSB compare with nsb2 and a simple empirical baseline?
3. **Physical interpretation:** what changes when Oxygen emission is included, and how sensitive are results to solar inputs?

Code: AGPL-3.0-only, Copyright (C) 2026 Vallés Puig, Ramon. Data retain source terms.


## 1. Prepare the Python environment

Select a fresh **CPython 3.12 or 3.13** Jupyter or Google Colab kernel and
**Run All**. The next cell installs **`nsb-py==0.2.0` from PyPI** and this
notebook's dependencies using the current kernel's Python. No NSB checkout,
GitHub installation of NSB or Rust toolchain is needed; a compatible NSB wheel
is required. The distribution is `nsb-py`, imported as `nsb`; the unrelated
package named `nsb` is rejected.

The package list is visible in `REQUIREMENTS`. Matching installed versions are
reused. If an already imported package changes, setup stops and asks you to
**restart the kernel and Run All**. Installation failures stop execution with
pip's error log; check network access and that you selected a supported kernel.

The **nsb2 reference** is installed from its fixed GitHub commit, so Git and
network access to GitHub are required for that package only. Its scientific
Python dependencies come from PyPI. Observations and reference data use public
services and verified local caches. The existing configuration below selects
cached inputs; leave registered scientific settings unchanged for reproduction.
Results remain inline, without JSON, checksum or figure exports.


In [ ]:
# Run this cell first, using a CPython 3.12 or 3.13 kernel.
import importlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys

if sys.implementation.name != "cpython" or sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("Select a CPython 3.12 or 3.13 kernel, then Run All.")

# nsb-py is the distribution; nsb is its Python import. Do not install 'nsb'.
REQUIREMENTS = ['nsb-py==0.2.0', 'numpy>=2,<3', 'matplotlib>=3.8,<4', 'astropy>=6.1,<9', 'scipy>=1.11,<2', 'healpy>=1.16', 'scikit-learn>=1.1,!=1.4.0', 'dust_extinction>=1.3']
try:
    metadata.distribution("nsb")
except metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("Remove the unrelated 'nsb' distribution with this kernel's pip, then restart.")

# Remember loaded dependencies so changes cannot silently leave stale imports.
loaded = {name: metadata.version(name)
          for module, names in metadata.packages_distributions().items()
          if module in sys.modules for name in names}
pip = [sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple"]
try:
    nsb_dist = metadata.distribution("nsb-py")
    direct = json.loads(nsb_dist.read_text("direct_url.json") or "{}")
except metadata.PackageNotFoundError:
    nsb_dist, direct = None, {}
if nsb_dist is not None and (nsb_dist.version != "0.2.0" or direct):
    if "nsb" in sys.modules:
        raise RuntimeError("Restart the kernel before selecting nsb-py==0.2.0 from PyPI.")
    subprocess.run(pip + ["--only-binary=:all:", "--no-deps", "--force-reinstall", "nsb-py==0.2.0"], check=True)
subprocess.run(pip + ["--only-binary=nsb-py", *REQUIREMENTS], check=True)
# The nsb2 reference is not on PyPI. Reuse the registered Git installation.
NSB2_COMMIT = "bc9320db03fbff69997ce366c6e76a37339c5d00"
try:
    reference = json.loads(metadata.distribution("nsb2").read_text("direct_url.json") or "{}")
except metadata.PackageNotFoundError:
    reference = {}
if (reference.get("url") != "https://github.com/GerritRo/nsb2.git"
        or reference.get("vcs_info", {}).get("commit_id") != NSB2_COMMIT):
    if "nsb2" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned nsb2 reference.")
    subprocess.run(pip + ["--force-reinstall", "--no-deps",
                          f"nsb2 @ git+https://github.com/GerritRo/nsb2.git@{NSB2_COMMIT}"], check=True)
importlib.invalidate_caches()
changed = [name for name, version in loaded.items() if metadata.version(name) != version]
if changed:
    raise RuntimeError(f"Updated already-imported packages: {changed}. Restart the kernel, then Run All.")

# Check that the module resolves to the nsb-py installation, rather than a local file.
nsb_dist = metadata.distribution("nsb-py")
nsb_spec = importlib.util.find_spec("nsb")
owned = {Path(nsb_dist.locate_file(f)).resolve() for f in nsb_dist.files or []}
if nsb_dist.version != "0.2.0" or nsb_spec is None or not nsb_spec.origin or Path(nsb_spec.origin).resolve() not in owned:
    raise RuntimeError("Expected the nsb-py==0.2.0 module. Remove shadowing nsb.py/directories and restart.")
print("Ready: nsb-py", nsb_dist.version)


In [ ]:
# Configure the optional location of existing inputs, then Restart and Run All.
import os
from pathlib import Path

CACHE = Path(os.environ.get("NSB_VALIDATION_CACHE", str(Path.home() / ".cache/nsb-observational")))

# Registered scientific settings.
REPETITIONS, SEED, PRACTICAL_MARGIN = 2000, 182, 0.10


In [ ]:
import hashlib
import importlib.metadata as metadata
import tarfile
from datetime import UTC, date, timedelta
from io import BytesIO
from itertools import combinations
from types import SimpleNamespace

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.constants import c, h
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.io import fits
from astropy.time import Time
from astropy.utils import iers
from astropy.utils.data import download_file
from scipy.integrate import simpson
from IPython.display import Markdown, display
import nsb
import nsb2

iers.conf.auto_download = False
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
LABEL = "Patat development experiment"
display(Markdown(f"**{LABEL}**"))
# Registered Paranal geometry and comparison definitions.
SITE = {"longitude": -70.4045, "latitude": -24.6272, "height": 2635.0}
MODELS = ("NSB continuum", "NSB Automatic", "nsb2")
COMPONENTS = ("zodiacal", "airglow", "starlight", "moonlight")
DENSITY_UNIT = 1 / u.ns / u.cm**2 / u.sr / u.nm


In [ ]:
versions = {name: metadata.version(name) for name in
            ("nsb-py", "nsb2", "numpy", "scipy", "astropy", "matplotlib", "siderust", "tempoch")}
display(Markdown(f"**Installed models:** nsb-py {versions['nsb-py']}; nsb2 {versions['nsb2']}."))


## 1. Observations and scope

The [Patat (2008) FORS1 Night Sky Spectral Library](https://www.eso.org/~fpatat/science/skybright/)
provides flux-calibrated sky spectra at Paranal. The registered selection contains
**103 spectra from 22 nights**: nine spectra on five nights define the empirical
baseline; the remaining **94 spectra on 17 nights** are used for error assessment.
Several spectra from one night share observing conditions and are not independent nights.

The sample applies the following selection criteria:

- **Observing configuration:** DARKFLG=0, grism 300V and filter FREE.
- **Astronomical night:** Sun altitude < −18°.
- **Moonless conditions:** Moon altitude < −5° (the lunar-altitude cut adopted for this sample).
- **Airmass:** 1–1.5.
- **Galactic latitude:** |b| ≥20°.
- **Exposure time:** ≥60 s.
- **Spectral coverage:** complete 365–650 nm coverage.

These cuts define a restricted dark-sky regime, rather than a survey of all
observing conditions. Integer MJD defines the registered night.

For an energy spectrum $I_E$, the measured photon radiance is
$N=\int I_E(\lambda)\lambda/(hc)\,d\lambda$ over 365–650 nm.
The integration retains noisy negative spectral samples and handles the band
edges explicitly; the resulting integrated observations must be positive.
The exact measurement and input checks are documented in the appendix.


In [ ]:
PATAT_SHA = "1f615fccb6dd464ab95b91f34adb4ba8ba3f4beebe97559204483c1da2c9965c"
PATAT_URLS = (
    "https://www.eso.org/~fpatat/science/skybright/DATA/skyspectra.tar",
    "https://web.archive.org/web/20160329190127id_/http://www.eso.org/~fpatat/science/skybright/DATA/skyspectra.tar",
)
SOLAR_SHA = "7240df111ea45f26057b89d3e9880a58856d7b6d040ad16247db41f5546dac96"
SOLAR_URL = (
    "https://www.ngdc.noaa.gov/stp/space-weather/solar-data/solar-features/"
    "solar-radio/noontime-flux/penticton/penticton_observed/listings/"
    "listing_drao_noontime-flux-observed_daily.txt"
)
CALIBRATION_NIGHTS = frozenset((51643, 51725, 51761, 51785, 52847))
# Exact SI constants expressed in cgs. 1 nm=10 A; 1 ns=1e-9 s.
HC_ERG_CM = 6.62607015e-27 * 2.99792458e10
ARCSEC2_PER_SR = (180 * 3600 / np.pi) ** 2
UNIT = "photons cm^-2 ns^-1 sr^-1"

COHORT_SHA = "3efdd3a0f3af19bfbc7615864f928b33fe22bcdbd6c67568fe317c059d44a1d6"


def verified_input(path, expected, urls):
    """Reuse an existing input or Astropy's download cache; verify its identity."""
    path = Path(path)
    if not path.exists():
        path = Path(download_file(
            urls[0], sources=list(urls), cache=True, show_progress=False, timeout=90
        ))
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Input checksum mismatch: {path}; expected {expected}, got {actual}")
    return path


In [ ]:
def measured_radiance(wavelength_angstrom, energy, lower=365.0, upper=650.0):
    """Exact integral of lambda times piecewise-linear I_E, retaining noisy negatives.

    erg/s/cm2/A/arcsec2 -> photons/cm2/ns/sr. On each clipped interval,
    Simpson integration is exact for the quadratic product lambda*I_E.
    This differs from trapezoid integration of pre-converted photon samples.
    """
    wave = np.asarray(wavelength_angstrom, float)
    flux = np.asarray(energy, float)
    if (
        wave.ndim != 1 or wave.size < 2 or flux.shape != wave.shape
        or not np.isfinite(wave).all() or not np.isfinite(flux).all()
        or np.any(np.diff(wave) <= 0)
        or not wave[0] <= lower * 10 < upper * 10 <= wave[-1]
    ):
        raise ValueError("Finite increasing full-coverage spectral samples required")
    edges = np.r_[lower * 10, wave[(wave > lower * 10) & (wave < upper * 10)], upper * 10]
    # Each Simpson panel spans one original linear segment, including clipped edges.
    grid = np.sort(np.r_[edges, (edges[:-1] + edges[1:]) / 2])
    area = simpson(grid * np.interp(grid, wave, flux), x=grid)
    return float(area * 1e-8 / HC_ERG_CM * ARCSEC2_PER_SR * 1e-9)


def photon_density(wave_angstrom, energy):
    """Per-nm photon density; diagnostic only, broadband uses exact operator above."""
    return np.asarray(energy) * np.asarray(wave_angstrom) * 1e-8 / HC_ERG_CM * ARCSEC2_PER_SR * 1e-8


def exclusion(row):
    """Frozen sequential selection, including DARKFLG; boundaries are intentional."""
    cuts = (
        (row["dark"] != 0, "dark_flag"),
        (row["grism"] != "300V", "grism"),
        (row["filter"] != "FREE", "filter"),
        (row["sun_alt"] >= -18, "twilight"),
        (row["moon_alt"] >= -5, "moon"),
        (not 1 <= row["airmass"] <= 1.5, "airmass"),
        (abs(row["gal_lat"]) < 20, "galactic_latitude"),
        (row["exposure"] < 60, "exposure"),
        (row["wave"][0] > 3650 or row["wave"][-1] < 6500, "coverage"),
    )
    return next((reason for reject, reason in cuts if reject), None)


def load_cohort(archive, *, enforce_registration=True):
    rows, rejected = [], {}
    with tarfile.open(archive) as tar:
        members = sorted(
            (m for m in tar.getmembers() if m.isfile() and m.name.startswith("fxSKY") and m.name.endswith(".fits")),
            key=lambda m: m.name,
        )
        if len(members) != 1189 or len({m.name for m in members}) != len(members):
            raise ValueError("Registered FITS inventory changed")
        for member in members:
            with fits.open(BytesIO(tar.extractfile(member).read())) as hdus:
                hdr, flux = hdus[0].header, np.asarray(hdus[0].data, float)
                if hdr["BUNIT"].strip() != "erg/s/cm2/A/arcsec2":
                    raise ValueError(f"Unexpected flux unit: {member.name}")
                wave = hdr["CRVAL1"] + (np.arange(hdr["NAXIS1"]) + 1 - hdr["CRPIX1"]) * hdr["CDELT1"]
                fields = {
                    "mjd": "MJD-OBS",
                    "ra": "RA",
                    "dec": "DEC",
                    "airmass": "AIRMASS",
                    "sun_alt": "ALTSUN",
                    "moon_alt": "ALTMOON",
                    "gal_lat": "GALAT",
                    "header_sfu": "SUNFLUX",
                    "exposure": "ORIGEXP",
                    "dark": "DARKFLG",
                }
                row = {key: float(hdr[value]) for key, value in fields.items()}
                if not np.isfinite(list(row.values())).all():
                    raise ValueError(f"Nonfinite metadata: {member.name}")
                row.update(
                    id=member.name,
                    night=int(row["mjd"]),
                    wave=wave,
                    energy=flux,
                    grism=hdr["GRISM"].strip(),
                    filter=hdr["FILTER"].strip(),
                )
            reason = exclusion(row)
            if reason:
                rejected[row["id"]] = reason
                continue
            row["observed"] = measured_radiance(wave, flux)
            row["split"] = "calibration" if row["night"] in CALIBRATION_NIGHTS else "evaluation"
            if not np.isfinite(row["observed"]) or row["observed"] <= 0:
                raise ValueError("Nonpositive integrated observational radiance")
            rows.append(row)
    identities = "\n".join(f"{r['id']}:{r['night']}:{r['split']}" for r in rows).encode()
    identity_sha = hashlib.sha256(identities).hexdigest()
    if enforce_registration:
        if identity_sha != COHORT_SHA or len(rows) != 103:
            raise ValueError(
                "Selected observation identities/night partition differ from registration"
            )
    return rows, {
        "selected": len(rows),
        "nights": len({r["night"] for r in rows}),
        "excluded_ids": rejected,
        "identity_sha256": identity_sha,
    }


### Coverage of the observing conditions

The next outputs show the calibration/evaluation partition, radiance across the
observing dates and the sampled airmass distribution. They establish the empirical
range tested here. Clustering in date or airmass limits how broadly a mean error
can be interpreted; the observations do not establish calibration at another site.

Solar-input availability is reported explicitly. An unavailable scenario is not
replaced by a different dataset or silently omitted from a complete assessment.


In [ ]:
def solar_windows(path, rows):
    """Centered means require all 27 observed daily values; inputs are pre-verified."""
    path = Path(path)
    daily = {}
    for line in path.read_text().splitlines():
        fields = line.split()
        if len(fields) == 2 and fields[1] != ".":
            daily[date(int(fields[0][:4]), int(fields[0][4:6]), int(fields[0][6:8]))] = float(
                fields[1]
            )
    values = {}
    for row in rows:
        center = Time(row["mjd"], format="mjd").to_datetime().date()
        days = [center + timedelta(days=i) for i in range(-13, 14)]
        if any(d not in daily for d in days):
            raise ValueError("Incomplete centered 27-day solar window")
        sample = np.array([daily[d] for d in days], float)
        if not np.isfinite(sample).all() or np.any(sample <= 0):
            raise ValueError("Invalid solar input")
        values[row["id"]] = float(np.mean(sample))
    return values


In [ ]:
availability, failures = {}, []
predictions, metrics, spectral = {}, {}, None
from astropy.utils import data as astropy_data

astropy_data.conf.allow_internet = True
archive = verified_input(CACHE / "skyspectra.tar", PATAT_SHA, PATAT_URLS)
rows, selection = load_cohort(archive)
if (
    len({r["night"] for r in rows}),
    sum(r["split"] == "evaluation" for r in rows),
    len({r["night"] for r in rows if r["split"] == "evaluation"}),
) != (22, 94, 17):
    raise ValueError("Registered night/evaluation counts changed")
availability["native"] = "available"
try:
    solar = verified_input(CACHE / "solar-observed.txt", SOLAR_SHA, (SOLAR_URL,))
    f27 = solar_windows(solar, rows)
    for row in rows:
        row["f27"] = f27[row["id"]]
    solar_info = {
        "convention": "observed noontime; centered 27 complete UTC days",
        "outside_palace_training": sum(not 67 <= r["f27"] <= 166 for r in rows),
    }
    availability["matched"] = "available"
except (OSError, ValueError, RuntimeError) as error:
    availability["matched"] = f"UNAVAILABLE: {error}"

evaluation = np.array([i for i, r in enumerate(rows) if r["split"] == "evaluation"])
y = np.array([r["observed"] for r in rows])[evaluation]
coverage = "| Partition | Spectra | Nights |\n|---|---:|---:|\n"
for split in ("calibration", "evaluation"):
    subset = [r for r in rows if r["split"] == split]
    coverage += f"| {split} | {len(subset)} | {len({r['night'] for r in subset})} |\n"
display(Markdown(coverage))
display(Markdown("**Scenario availability:** " + str(availability)))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for split in ("calibration", "evaluation"):
    subset = [r for r in rows if r["split"] == split]
    axes[0].scatter([r["mjd"] for r in subset], [r["observed"] for r in subset], label=split, s=18)
axes[0].set(xlabel="Observation time [MJD UTC]", ylabel=f"Observed [{UNIT}]")
axes[0].legend()
axes[1].hist([r["airmass"] for r in rows], bins=12)
axes[1].set(xlabel="Header airmass [dimensionless]", ylabel="Spectra")
fig.suptitle(f"{LABEL}: 365–650 nm coverage")
plt.show()


## 2. Comparison design

All configurations predict the same band-integrated observable for the same
observing times and directions. No telescope response is applied.

| Configuration | Role in the assessment |
|---|---|
| **NSB Automatic (Oxygen)** | nsb-py 0.2.0 default: PALACE continuum plus experimental Oxygen emission |
| **NSB continuum** | Controlled comparison with Oxygen excluded; other NSB assumptions remain fixed |
| **nsb2** | External implementation using Gaia starlight, Leinert zodiacal light and combined ESO SkyCalc Airglow |
| **Calibration baseline** | Constant median of the nine calibration spectra; tests whether physical predictions improve on a local empirical reference |

**Solar inputs.** **Native** evaluates each
model with its own solar resolver: NSB's solar history and nsb2's Patat SUNFLUX
rounded to 0.1 sfu. **Matched** supplies an independently measured, centered
27-day observed F10.7 mean from NRC/Penticton via NOAA. It examines sensitivity
to a common solar input; it is not a refit to the sky observations. Both scenarios
are reported separately.

The matched nsb2 Airglow follows the registered rescaling
`(0.2 + 0.00614 × measured F27)/(0.2 + 0.00614 × native SUNFLUX)`.
PALACE's solar training range is **67–166 sfu**. Extrapolations are retained and
identified; observed/adjusted solar-convention equivalence is not assumed.

**Starlight spectral assumption.** NSB's integrated Starlight map lacks
an admitted restricted-band spectral product. Its 365–650 nm contribution uses
the registered Rieke solar-photon fraction of the 300–650 nm map, a research-only
spectral proxy. nsb2's faint Gaia population (G≥15), propagation and Airglow
content differ from NSB's. Component names therefore do not imply identical
physical definitions. This experiment evaluates conditional research totals and
does not validate production spectral Starlight.


In [ ]:
def reference_integral(wave, density, lower, upper):
    """Frozen nsb2/Rieke protocol: trapezoid on in-band source-grid nodes.

    Retained to compare with original evidence (not a new integration rule).
    The exact measured-energy operator is separate and inserts band edges.
    """
    wave, density = np.asarray(wave, float), np.asarray(density, float)
    if (
        wave.shape != density.shape
        or wave.ndim != 1
        or not np.isfinite(density).all()
        or not np.isfinite(wave).all()
        or np.any(np.diff(wave) <= 0)
        or wave[0] > lower
        or wave[-1] < upper
    ):
        raise ValueError("Invalid source spectrum or incomplete coverage")
    mask = (wave >= lower) & (wave <= upper)
    if np.count_nonzero(mask) < 2:
        raise ValueError("Insufficient in-band source-grid nodes")
    return float(np.trapezoid(density[mask], wave[mask]))


def starlight_fraction():
    from nsb2.core.photometry import SolarSpectrumRieke2008

    wave, energy = SolarSpectrumRieke2008()
    photon = (energy * wave / (h * c)).to_value(1 / u.s / u.cm**2 / u.nm)
    wave = wave.to_value(u.nm)
    return reference_integral(wave, photon, 365, 650) / reference_integral(wave, photon, 300, 650)


def prepare_nsb2_reference_assets():
    """Cache the published nsb2 assets using Astropy's downloader."""
    from nsb2.core.photometry import SOLAR_SPECTRUM_URL
    from nsb2.core.spectral import CALSPEC_URL, SVO_TABLE_URL, VEGA_CALSPEC_FILE
    from nsb2.emitter.stars import GAIA_MAP_URL

    inputs = (
        ("Rieke solar spectrum (STScI)", SOLAR_SPECTRUM_URL),
        ("Faint Gaia DR3 map (Zenodo)", GAIA_MAP_URL),
        ("Gaia G passband (SVO)", SVO_TABLE_URL + "GAIA/GAIA3.G"),
        ("Gaia Gbp passband (SVO)", SVO_TABLE_URL + "GAIA/GAIA3.Gbp"),
        ("Gaia Grp passband (SVO)", SVO_TABLE_URL + "GAIA/GAIA3.Grp"),
        ("Vega CALSPEC reference (STScI)", CALSPEC_URL + VEGA_CALSPEC_FILE),
    )
    for label, url in inputs:
        print(f"Preparing {label}", flush=True)
        download_file(url, cache=True, show_progress=False, timeout=90)


In [ ]:
class Models:
    def __init__(self):
        import siderust
        from nsb2.emitter.stars import from_gaia_dr3_map
        from nsb2.emitter.zodiacal import from_leinert1998

        import nsb

        self.nsb, self.siderust = nsb, siderust
        self.observer = siderust.Observer(
            lon_deg=SITE["longitude"], lat_deg=SITE["latitude"], height_m=SITE["height"]
        )
        self.site = EarthLocation.from_geodetic(
            SITE["longitude"] * u.deg, SITE["latitude"] * u.deg, SITE["height"] * u.m
        )
        from astropy.utils import data as astropy_data
        astropy_data.conf.remote_timeout = 90
        print(
            "Preparing nsb2 reference assets (published upstream URLs; Astropy cache)",
            flush=True,
        )
        prepare_nsb2_reference_assets()
        self.fraction = starlight_fraction()
        self.zodiacal = from_leinert1998()
        self.stars = from_gaia_dr3_map()
        self.air_sources = {}

    def query(self, row, mask):
        return self.nsb.PointQuery(
            self.observer,
            Time(row["mjd"], format="mjd", scale="utc").to_datetime(timezone=UTC),
            self.siderust.Direction(ra_deg=row["ra"], dec_deg=row["dec"]),
            components=mask,
        )

    def nsb_components(self, row, oxygen, sfu=None):
        nsb = self.nsb
        # Automatic == Oxygen for v0.2.0; continuum must be selected explicitly.
        if oxygen:
            config = nsb.NsbModelConfig.generic_clear_sky()  # Automatic → Oxygen
        else:
            config = nsb.NsbModelConfig.generic_clear_sky().with_airglow_model(
                nsb.AirglowModel.PARANAL_PALACE_V1
            )
        if sfu is not None:
            config = config.with_solar_radio_flux_sfu(float(sfu))
        evaluator = nsb.NsbEvaluator(config)
        # Includes Moon explicitly: API must establish physical zero, not a fabricated band.
        mask = nsb.ComponentMask.ZODIACAL | nsb.ComponentMask.AIRGLOW | nsb.ComponentMask.MOON
        band = evaluator.integrate_band_components(self.query(row, mask), 365.0, 650.0)
        result = {
            component.name: float(component.integrated_photons_cm2_ns_sr)
            for component in band.components
        }
        # Runtime uses "moon" rather than the report's "moonlight" label.
        result["moonlight"] = result.pop("moon")
        if result["moonlight"] != 0.0:
            raise ValueError("Registered moonless assumption violated")
        star = evaluator.evaluate(self.query(row, nsb.ComponentMask.STARLIGHT))
        result["starlight"] = float(star.components[0].integrated_photons_cm2_ns_sr) * self.fraction
        return {key: result[key] for key in COMPONENTS}

    def source_integral(self, source, row):
        frame = AltAz(
            obstime=Time(row["mjd"], format="mjd", scale="utc"),
            location=self.site,
            pressure=0 * u.hPa,
        )
        direction = SkyCoord(ra=[row["ra"]] * u.deg, dec=[row["dec"]] * u.deg)
        field, _ = source.query_direct(SimpleNamespace(origin=frame), direction, np.zeros(1))
        wave = field.spectral_grid.wvl.to_value(u.nm)
        raw = field.spectral_grid(field.spectral_data)
        try:
            density = (raw[:, 0] * field.weights[0, 0]).to_value(DENSITY_UNIT)
        except u.UnitConversionError:
            density = None
        if density is None or density.shape != wave.shape:
            # The public resolver handles the source's photometric normalization units.
            from nsb2.core.spectral import Bandpass

            band = Bandpass(np.array([365.0, 400.0, 550.0, 650.0]) * u.nm, np.ones(4))
            rate = field.resolve_spectra(band).integrate()
            return (
                float(
                    np.asarray(
                        rate.to_value(
                            1 / u.s / u.cm**2 / u.sr, equivalencies=u.dimensionless_angles()
                        )
                    ).reshape(-1)[0]
                )
                * 1e-9
            )
        return reference_integral(wave, density, 365, 650)

    def nsb2_components(self, row, sfu=None):
        from nsb2.emitter.airglow import from_eso_skycalc

        # Native nsb2 protocol uses SUNFLUX rounded to 0.1 sfu. Matched diagnostic
        # scales that exact source by its scalar law, with unrounded measured f27.
        native_flux = round(row["header_sfu"], 1)
        if native_flux not in self.air_sources:
            self.air_sources[native_flux] = from_eso_skycalc(90 * u.km, native_flux)
        air = self.source_integral(self.air_sources[native_flux], row)
        if sfu is not None:
            air *= (0.2 + 0.00614 * sfu) / (0.2 + 0.00614 * native_flux)
        return {
            "zodiacal": self.source_integral(self.zodiacal, row),
            "airglow": air,
            "starlight": self.source_integral(self.stars, row),
            "moonlight": 0.0,
        }

    def predict(self, row, model, sfu=None):
        parts = (
            self.nsb2_components(row, sfu)
            if model == "nsb2"
            else self.nsb_components(row, model == "NSB Automatic", sfu)
        )
        if (
            set(parts) != set(COMPONENTS)
            or not np.isfinite(list(parts.values())).all()
            or min(parts.values()) < 0
        ):
            raise ValueError("Invalid/incomplete component prediction")
        return {"id": row["id"], "total": float(sum(parts.values())), "components": parts}

    def spectral_example(self, row, sfu=None):
        """Actual restricted API bin integrals, never an invented resolved line profile."""
        nsb = self.nsb
        bins = np.arange(550.0, 645.0, 1.0)
        result = {
            "id": row["id"],
            "air_wavelength_nm": (row["wave"] / 10).tolist(),
            "observed_total_density": photon_density(row["wave"], row["energy"]).tolist(),
            "vacuum_bin_edges_nm": bins.tolist(),
            "scenario": "native" if sfu is None else "matched",
        }
        for name, model in (
            ("continuum", nsb.AirglowModel.PARANAL_PALACE_V1),
            ("automatic", None),
            ("oxygen", nsb.AirglowModel.PARANAL_PALACE_OXYGEN_V1),
        ):
            cfg = nsb.NsbModelConfig.generic_clear_sky()
            if model is not None:
                cfg = cfg.with_airglow_model(model)
            if sfu is not None:
                cfg = cfg.with_solar_radio_flux_sfu(sfu)
            ev = nsb.NsbEvaluator(cfg)
            result[name] = [
                float(
                    ev.integrate_band(
                        self.query(row, nsb.ComponentMask.AIRGLOW), float(a), float(b)
                    )
                )
                for a, b in zip(bins[:-1], bins[1:], strict=True)
            ]
        return result


In [ ]:
models = Models()
for scenario, status in availability.items():
    if status != "available":
        continue
    predictions[scenario] = {m: [] for m in MODELS}
    for i, row in enumerate(rows):
        for model in MODELS:
            try:
                predictions[scenario][model].append(
                    models.predict(row, model, row["f27"] if scenario == "matched" else None)
                )
            except Exception as error:
                failures.append(
                    dict(scenario=scenario, model=model, id=row["id"], reason=str(error))
                )
        if (i + 1) % 25 == 0 or i + 1 == len(rows):
            print(f"{scenario}: {i + 1}/{len(rows)} spectra")
spectral = models.spectral_example(rows[0], rows[0].get("f27"))
display(
    Markdown(
        f"**Rejected predictions: {len(failures)}** (IDs and reasons remain in `failures`)."
    )
)


### How performance and uncertainty are measured

Residual = prediction − observation: positive bias means a sky prediction that is
too bright. **Mean absolute relative error (relative MAE)** measures the mean
of $|N_{pred}-N_{obs}|/N_{obs}$; **relative bias** is the mean of
$(N_{pred}-N_{obs})/N_{obs}$.
Both are displayed as percentages. MAE and RMSE retain radiance units. Each
spectrum has equal weight, regardless of exposure duration.

The baseline is fitted only on the calibration nights. For uncertainty we resample
**whole evaluation nights**, pairing the same draws across all configurations:
2,000 repetitions with seed 182. The percentile 95% intervals describe night-sampling
variability conditional on the fixed baseline, and exclude shared calibration systematics.
They are not prediction intervals for a new observation.

For paired relative-MAE differences, negative **NSB minus reference** favors NSB;
positive favors the reference. Differences are shown in **percentage points (pp)**.
An interval crossing zero establishes neither a difference nor equivalence.
The registered practical margin is **10 pp**: a practical directional claim
requires the entire paired interval beyond that margin.

Every model must predict all 103 spectra before a scenario can be scored on its
94 evaluation spectra. Missing or invalid predictions invalidate that scenario;
no subset is selected to improve agreement.


In [ ]:
def align(rows, predictions):
    """Exact identifier matching; reject duplicates, extras, missing/nonfinite predictions."""
    ids = [r["id"] for r in rows]
    if not ids or len(set(ids)) != len(ids):
        raise ValueError("Unique nonempty observation identities required")
    result = {}
    missing = {}
    for model, records in predictions.items():
        mapping = {r["id"]: r for r in records}
        missing[model] = sorted(set(ids) - set(mapping))
        if len(mapping) != len(records) or set(mapping) != set(ids):
            raise ValueError(f"{model}: cohort mismatch; missing={missing[model]}")
        values = np.array([mapping[key]["total"] for key in ids], float)
        if not np.isfinite(values).all() or np.any(values < 0):
            raise ValueError(f"{model}: invalid predictions")
        result[model] = values
    return result


def baseline(rows):
    train = [r for r in rows if r["split"] == "calibration"]
    test = [r for r in rows if r["split"] == "evaluation"]
    if not train or not test or set(r["night"] for r in train) & set(r["night"] for r in test):
        raise ValueError("Disjoint nonempty calibration and evaluation nights required")
    return float(np.median([r["observed"] for r in train]))


def errors(observed, predicted):
    y, p = np.asarray(observed, float), np.asarray(predicted, float)
    if (
        y.ndim != 1
        or y.size == 0
        or y.shape != p.shape
        or not np.isfinite(y).all()
        or not np.isfinite(p).all()
        or np.any(y <= 0)
    ):
        raise ValueError("Finite aligned vectors and positive observations required")
    d = p - y
    rel = d / y
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        values = {
            "mean_signed_error": np.mean(d),
            "mae": np.mean(abs(d)),
            "rmse": np.linalg.norm(d / np.sqrt(len(d))),
            "median_absolute_error": np.median(abs(d)),
            "mean_absolute_relative_error": np.mean(abs(rel)),
            "median_absolute_relative_error": np.median(abs(rel)),
            "relative_bias": np.mean(rel),
            "relative_rmse": np.linalg.norm(rel / np.sqrt(len(rel))),
        }
    if not np.isfinite(list(values.values())).all():
        raise ValueError("Unstable metric values")
    return {k: float(v) for k, v in values.items()}


def scores(observed, predictions, nights, repetitions=2000, seed=182):
    y, nights = np.asarray(observed, float), np.asarray(nights)
    if nights.shape != y.shape or repetitions < 100 or not predictions:
        raise ValueError("Matching nights, models and >=100 repetitions required")
    groups = [np.flatnonzero(nights == n) for n in np.unique(nights)]
    if len(groups) < 3:
        raise ValueError("At least three independent nights required for intervals")
    predictions = {k: np.asarray(v, float) for k, v in predictions.items()}
    metrics = {
        k: {**errors(y, p), "n": len(y), "nights": len(groups), "missing": 0}
        for k, p in predictions.items()
    }
    # Resample whole nights with shared draws; divide by spectrum counts, not nights.
    indices = np.random.default_rng(seed).integers(len(groups), size=(repetitions, len(groups)))
    counts = np.array([len(g) for g in groups])
    denominator = counts[indices].sum(axis=1)[:, None]
    draws = {}
    for key, p in predictions.items():
        delta = abs(p - y)
        sums = np.array([[delta[g].sum(), (delta[g] / y[g]).sum()] for g in groups])
        draws[key] = sums[indices].sum(axis=1) / denominator
    for key in predictions:
        metrics[key]["mae_ci95"] = np.percentile(draws[key][:, 0], [2.5, 97.5]).tolist()
        metrics[key]["relative_mae_ci95"] = np.percentile(draws[key][:, 1], [2.5, 97.5]).tolist()
    paired = {}
    for a, b in combinations(predictions, 2):
        paired[f"{a}_minus_{b}"] = {
            "mae_difference": metrics[a]["mae"] - metrics[b]["mae"],
            "relative_mae_difference": metrics[a]["mean_absolute_relative_error"]
            - metrics[b]["mean_absolute_relative_error"],
            "mae_ci95": np.percentile(draws[a][:, 0] - draws[b][:, 0], [2.5, 97.5]).tolist(),
            "relative_mae_ci95": np.percentile(
                draws[a][:, 1] - draws[b][:, 1], [2.5, 97.5]
            ).tolist(),
        }
    return {
        "models": metrics,
        "paired": paired,
        "repetitions": repetitions,
        "seed": seed,
        "estimand": "equal weight per spectrum (historically called exposure-weighted); resample whole nights",
        "interval_scope": "sampling variability conditional on fixed calibration baseline; no calibration-systematic uncertainty",
    }


In [ ]:
baseline_value = baseline(rows)
for scenario, records in predictions.items():
    try:
        vectors = align(rows, records)
        if set(vectors) != set(MODELS):
            raise ValueError("Required model missing")
    except ValueError as error:
        availability[scenario] = f"INVALID: {error}; no intersection scoring"
        continue
    vectors["baseline"] = np.full(len(rows), baseline_value)
    metrics[scenario] = scores(
        y,
        {k: v[evaluation] for k, v in vectors.items()},
        [rows[i]["night"] for i in evaluation],
        REPETITIONS,
        SEED,
    )
    for pair, p in metrics[scenario]["paired"].items():
        lo, hi = p["relative_mae_ci95"]
        decision = (
            "no demonstrated difference"
            if lo <= 0 <= hi
            else "left practically lower"
            if hi < -PRACTICAL_MARGIN
            else "right practically lower"
            if lo > PRACTICAL_MARGIN
            else "direction detected; practical margin not established"
        )
        p["interpretation"] = decision


## 3. Measured performance

The tables below give the numerical assessment for each solar scenario. Read the
mean errors alongside their night-based uncertainty and the **paired** comparisons;
a smaller point estimate alone does not establish superiority. The local baseline
is included even when it outperforms a physical model.


In [ ]:
DISPLAY_NAMES = {"NSB Automatic": "NSB Automatic (Oxygen)", "NSB continuum": "NSB continuum", "nsb2": "nsb2", "baseline": "Calibration baseline"}
COLORS = {"NSB Automatic": "#0072B2", "NSB continuum": "#D55E00", "nsb2": "#009E73", "baseline": "#777777"}
SCENARIOS = {"native": "Native solar inputs", "matched": "Matched measured F10.7"}


def paired_comparison(result, left, right):
    # Preserve the computed pair order; invert only the displayed direction.
    direct = f"{left}_minus_{right}"
    reverse = f"{right}_minus_{left}"
    if direct in result["paired"]:
        p = result["paired"][direct]
        return p["relative_mae_difference"], np.array(p["relative_mae_ci95"])
    p = result["paired"][reverse]
    return -p["relative_mae_difference"], -np.array(p["relative_mae_ci95"])[::-1]


def comparison_text(difference, interval, reference):
    lo, hi = interval
    if lo <= 0 <= hi:
        finding = "no difference established; equivalence is not established either"
    elif hi < 0:
        finding = "lower NSB error"
    else:
        finding = "higher NSB error"
    practical = "registered 10 pp practical margin established" if hi < -PRACTICAL_MARGIN or lo > PRACTICAL_MARGIN else "registered 10 pp practical margin not established"
    return (f"**NSB Automatic − {reference}: {100*difference:+.2f} pp** "
            f"(paired 95% CI {100*lo:+.2f} to {100*hi:+.2f} pp): {finding}; {practical}.")


if set(metrics) != {"native", "matched"} or failures:
    display(Markdown("**Incomplete assessment:** valid scenario results below are descriptive only. No overall comparative conclusion is drawn."))

for scenario in ("native", "matched"):
    if scenario not in metrics:
        display(Markdown(f"### {SCENARIOS[scenario]}: unavailable or invalid\n\n{availability.get(scenario, 'Not evaluated')}. No performance conclusion is drawn."))
        continue
    result = metrics[scenario]
    table = f"### {SCENARIOS[scenario]}\n\n"
    table += "| Model | Relative MAE [95% night CI] | Relative bias | MAE | RMSE |\n|---|---:|---:|---:|---:|\n"
    for name in ("NSB Automatic", "nsb2", "NSB continuum", "baseline"):
        m = result["models"][name]
        lo, hi = m["relative_mae_ci95"]
        table += (f"| {DISPLAY_NAMES[name]} | {100*m['mean_absolute_relative_error']:.2f}% "
                  f"[{100*lo:.2f}, {100*hi:.2f}]% | {100*m['relative_bias']:+.2f}% | "
                  f"{m['mae']:.5f} | {m['rmse']:.5f} |\n")
    display(Markdown(table + f"\nMAE and RMSE: {UNIT}. Relative bias is a point estimate; its interval is not estimated by this protocol."))
    m = result["models"]["NSB Automatic"]
    display(Markdown(f"NSB Automatic's mean absolute relative error is **{100*m['mean_absolute_relative_error']:.2f}%** across **{m['n']} spectra / {m['nights']} nights**. Its signed relative bias is **{100*m['relative_bias']:+.2f}%**, indicating {'overprediction' if m['relative_bias'] > 0 else 'underprediction' if m['relative_bias'] < 0 else 'no mean signed relative offset'}."))
    for reference in ("nsb2", "baseline", "NSB continuum"):
        difference, interval = paired_comparison(result, "NSB Automatic", reference)
        display(Markdown(comparison_text(difference, interval, DISPLAY_NAMES[reference])))


### Agreement with the measured sky

Points on the diagonal reproduce the observations; points below it underpredict
the sky. The dotted horizontal line is the fitted calibration baseline. Shared
residual bins make biases and error tails comparable across models: zero is exact
agreement, negative values are too faint and positive values are too bright.


In [ ]:
def show_figure(fig, name):
    fig.suptitle(f"{LABEL}: {name}", fontsize=11)
    plt.show()


def evaluation_values(scenario):
    return {m: np.array([r["total"] for r in predictions[scenario][m]])[evaluation] for m in MODELS}


for scenario, result in metrics.items():
    values = evaluation_values(scenario)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained")
    for model, predicted in values.items():
        axes[0].scatter(y, predicted, s=18, alpha=0.65, color=COLORS[model], label=DISPLAY_NAMES[model])
    high = max(y.max(), *(p.max() for p in values.values())) * 1.05
    axes[0].plot([0, high], [0, high], color="black", lw=1, label="Perfect agreement")
    axes[0].axhline(baseline_value, color=COLORS["baseline"], ls=":", label="Calibration baseline")
    axes[0].set(xlabel=f"Observed [{UNIT}]", ylabel=f"Predicted [{UNIT}]", title="Agreement with observations", xlim=(0, high), ylim=(0, high))
    axes[0].legend(fontsize=8)
    residuals = {m: 100 * (p - y) / y for m, p in values.items()}
    baseline_residual = 100 * (baseline_value - y) / y
    bins = np.linspace(min(baseline_residual.min(), *(r.min() for r in residuals.values())), max(baseline_residual.max(), *(r.max() for r in residuals.values())), 22)
    for model, residual in {**residuals, "baseline": baseline_residual}.items():
        axes[1].hist(residual, bins=bins, histtype="step", color=COLORS[model], label=DISPLAY_NAMES[model])
    axes[1].axvline(0, color="black", lw=1)
    axes[1].set(xlabel="Signed relative residual [%]", ylabel="Evaluation spectra", title="Direction and spread of errors")
    axes[1].legend(fontsize=8)
    show_figure(fig, f"{SCENARIOS[scenario]}: observational agreement")


### Uncertainty in model differences

The left panel compares error magnitudes with 95% night-sampling intervals. The
right panel directly tests paired differences for NSB Automatic against nsb2,
the baseline and continuum. Use these paired intervals, rather than overlap of
individual-model intervals, to assess differences. Negative favors NSB Automatic;
zero means equal error in that draw. Dotted lines mark the registered ±10 pp margin.


In [ ]:
for scenario, result in metrics.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    names = ("NSB Automatic", "nsb2", "NSB continuum", "baseline")
    for j, name in enumerate(names):
        m = result["models"][name]
        axes[0].plot(100*np.array(m["relative_mae_ci95"]), [j, j], color=COLORS[name], lw=2)
        axes[0].scatter(100*m["mean_absolute_relative_error"], j, color=COLORS[name])
    axes[0].set(yticks=range(len(names)), yticklabels=[DISPLAY_NAMES[m] for m in names], xlabel="Mean absolute relative error [%]", title="Model error with 95% night intervals")
    axes[0].invert_yaxis()
    references = ("nsb2", "baseline", "NSB continuum")
    for j, reference in enumerate(references):
        delta, interval = paired_comparison(result, "NSB Automatic", reference)
        axes[1].plot(100*interval, [j, j], color=COLORS[reference], lw=2)
        axes[1].scatter(100*delta, j, color=COLORS[reference])
    axes[1].axvline(0, color="black", lw=1)
    axes[1].axvline(-100*PRACTICAL_MARGIN, color="grey", ls=":")
    axes[1].axvline(100*PRACTICAL_MARGIN, color="grey", ls=":")
    axes[1].set(yticks=range(len(references)), yticklabels=[f"NSB − {DISPLAY_NAMES[r]}" for r in references], xlabel="Relative-MAE difference [pp]; negative favors NSB", title="Paired 95% intervals; dotted lines: ±10 pp")
    axes[1].invert_yaxis()
    show_figure(fig, f"{SCENARIOS[scenario]}: comparative performance")


## 4. Physical interpretation and diagnostics

### Variation across nights and the effect of Oxygen

Night means reveal whether a bias persists or is concentrated in a few observing
nights; connecting lines are visual guides, not an interpolation of missing nights.
The per-spectrum bars isolate the change from continuum to Oxygen: blue indicates
reduced absolute error, orange increased error. A mean improvement need not benefit
every observation.

The component budget then places the modeled sources beside the mean observed
total. It helps locate differences in the radiance budget, but component definitions
and populations differ across implementations. It is not a component-by-component
observational validation.


In [ ]:
for scenario in metrics:
    values = evaluation_values(scenario)
    residuals = {m: (p-y)/y for m,p in values.items()}
    nights = np.array([rows[i]["night"] for i in evaluation])
    unique_nights = np.unique(nights)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
    for model, residual in residuals.items():
        axes[0].plot(unique_nights, [100*np.mean(residual[nights == night]) for night in unique_nights], marker="o", markersize=3, color=COLORS[model], label=DISPLAY_NAMES[model])
    axes[0].axhline(0, color="black", lw=1)
    axes[0].set(xlabel="Night [integer MJD]", ylabel="Night mean signed relative residual [%]", title="Does the bias persist across nights?")
    axes[0].legend(fontsize=8)
    delta = 100*(abs(residuals["NSB continuum"])-abs(residuals["NSB Automatic"]))
    axes[1].bar(range(len(y)), delta, color=np.where(delta >= 0, COLORS["NSB Automatic"], COLORS["NSB continuum"]), width=1)
    axes[1].axhline(0, color="black", lw=1)
    axes[1].set(xlabel="Evaluation spectrum index (registered ID order)", ylabel="Absolute relative error reduction [pp]", title="Oxygen: positive improves; negative degrades")
    show_figure(fig, f"{SCENARIOS[scenario]}: night variation and Oxygen effect")
    fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
    bottom = np.zeros(len(MODELS))
    for part in COMPONENTS:
        means = np.array([np.mean([predictions[scenario][m][i]["components"][part] for i in evaluation]) for m in MODELS])
        ax.bar([DISPLAY_NAMES[m] for m in MODELS], means, bottom=bottom, label=part)
        bottom += means
    ax.axhline(np.mean(y), color="black", ls="--", label="Mean observed total")
    ax.set(ylabel=f"Mean radiance [{UNIT}]", title="Component budgets under each model's assumptions")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
    show_figure(fig, f"{SCENARIOS[scenario]}: physical contributions")


### Oxygen spectral features and solar-input sensitivity

The spectral example uses the **first spectrum in the registered cohort order**,
not a spectrum selected for favorable agreement. It shows why including Oxygen can
change the broadband prediction, rather than measuring a spectral goodness of fit.
Observed **total** radiance is on air wavelengths; modeled **Airglow only** uses
vacuum-bin integrals divided by the 1 nm bin width. Other sky components, wavelength
conventions and instrumental resolution prevent a direct line-profile comparison.

The separate solar panel shows the change in absolute relative error between native
and matched inputs for every evaluation spectrum. Positive values mean the matched
input reduces error; negative values mean it increases error. The shaded 67–166 sfu
region is PALACE's training range. Viewing conditions covary with solar activity,
so this plot does not identify a causal solar effect or validate extrapolation.


In [ ]:
if spectral:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
    edges = np.array(spectral["vacuum_bin_edges_nm"])
    for ax, limits in zip(axes, ((552, 563), (625, 642)), strict=True):
        ax.plot(
            spectral["air_wavelength_nm"],
            spectral["observed_total_density"],
            color="black",
            label="Observed TOTAL; air wavelength",
        )
        for name in ("continuum", "oxygen"):
            ax.stairs(
                np.array(spectral[name]) / np.diff(edges),
                edges,
                label=f"NSB AIRGLOW {name}; vacuum bins",
                color=COLORS["NSB continuum" if name == "continuum" else "NSB Automatic"],
            )
        ax.set(xlim=limits, xlabel="Wavelength [nm]", ylabel=f"Radiance density [{UNIT} nm^-1]")
    axes[0].legend(fontsize=7)
    show_figure(fig, f"Spectral component illustration ({spectral['id']}; {spectral['scenario']})")
    if {"native", "matched"} <= set(metrics):
        fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
        f27 = np.array([rows[i]["f27"] for i in evaluation])
        ax.axvspan(67, 166, alpha=0.12, color="green", label="PALACE training domain")
        for model in MODELS:
            native = np.array([r["total"] for r in predictions["native"][model]])[evaluation]
            matched = np.array([r["total"] for r in predictions["matched"][model]])[evaluation]
            ax.scatter(f27, 100 * (abs(native - y) - abs(matched - y)) / y, s=18, color=COLORS[model], label=DISPLAY_NAMES[model])
        ax.axhline(0, color="black", lw=1)
        ax.set(
            xlabel="Observed centered 27-day F10.7 [sfu]",
            ylabel="Absolute relative error reduction [pp]",
        )
        ax.legend(fontsize=7)
        show_figure(fig, "Solar-input sensitivity: positive favors matched inputs")
    else:
        display(Markdown("**Solar sensitivity unavailable:** both native and matched scenarios must be valid."))
    if "matched" in availability and availability["matched"] == "available":
        display(
            Markdown(
                f"**{solar_info['outside_palace_training']}/103 observations outside PALACE's solar training range.**"
            )
        )


## 5. Conclusions and limits of the evidence


In [ ]:
complete_campaign = set(metrics) == {"native", "matched"} and not failures
if not complete_campaign:
    display(Markdown("**Incomplete assessment.** The two-scenario campaign is not complete. Valid scenario results above are descriptive only; no overall comparative conclusion is drawn. Inspect availability and failures in the appendix."))
else:
    display(Markdown("Both solar scenarios are complete. The following conclusions refer to this development cohort and its stated spectral assumptions."))
    for scenario in ("native", "matched"):
        result = metrics[scenario]
        display(Markdown(f"### {SCENARIOS[scenario]}"))
        for reference in ("nsb2", "baseline", "NSB continuum"):
            difference, interval = paired_comparison(result, "NSB Automatic", reference)
            display(Markdown(comparison_text(difference, interval, DISPLAY_NAMES[reference])))
        values = evaluation_values(scenario)
        reduction = abs(values["NSB continuum"]-y)-abs(values["NSB Automatic"]-y)
        improved, degraded = int(np.sum(reduction > 0)), int(np.sum(reduction < 0))
        display(Markdown(f"Including Oxygen reduces absolute error in **{improved}/{len(y)}** evaluation spectra, increases it in **{degraded}/{len(y)}**, and leaves **{int(np.sum(reduction == 0))}** unchanged. This describes a controlled model change; it does not independently validate the Oxygen model."))


### Findings and limitations

This comparison quantifies agreement with a restricted Paranal dark-time cohort.
It tests the practical consequences of Oxygen and solar-input choices under the
registered assumptions. A baseline with lower error is a meaningful result:
a physical model must demonstrate added predictive value beyond local calibration.

**Development dependence.** Patat observations informed Oxygen development.
Separating calibration and evaluation nights protects the baseline fit, but does
not make the evaluation partition independent of model development. An independent
post-development cohort with compatible diffuse calibration is needed to assess
generalization. These results establish neither CTAO site calibration nor global
sky accuracy, and do not test bright Moon, twilight or the full default band.

**Physical comparability.** Missing non-oxygen emissions and molecular absorption,
different stellar populations, the research-only Starlight spectral proxy and
differing propagation conventions limit physical equivalence. Component budget
differences cannot uniquely identify the cause of a total-radiance discrepancy.

**Uncertainty.** FORS flux calibration has shared few-percent uncertainty; no
per-spectrum statistical uncertainty is supplied. Night bootstrap intervals cover
sampling variability, not these systematics or a complete model uncertainty budget.
An interval crossing zero establishes neither superiority nor equivalence.
Solar extrapolations and night-to-night patterns should be considered alongside
mean errors before using this assessment for observational planning.


## Technical appendix: inputs and reproducibility

All results remain in this kernel: `rows` contains observations, `predictions`
contains the model outputs, `metrics` contains errors and paired intervals, and
`availability` and `failures` describe any incomplete scenarios. The notebook
writes no result files; tables and figures are displayed inline.

- **Input identity:** Patat spectra and solar data are checked against their
  registered SHA-256 values. Downloads use Astropy's cache; existing files under
  `CACHE` are reused after verification. FITS are read directly from the TAR,
  without extracting a directory. The cohort checksum fixes IDs, nights and splits.
- **Measurement:** SciPy Simpson integration with interval midpoints integrates
  wavelength times the piecewise-linear energy spectrum, retaining noisy negative
  samples. Unit conversion is explicit. The nsb2/Rieke reference continues to use
  NumPy trapezoids on its in-band grid nodes, a distinct measurement operator.
- **Solar coverage:** matched inputs require all 27 observed daily values, with
  no interpolation, imputation or fit. Changed input bytes or missing days disable
  the matched scenario rather than substitute other data.
- **Statistical design:** NumPy resamples whole nights with shared random draws.
  Error sums are divided by the number of spectra in each draw, preserving equal
  weight per spectrum. The baseline remains fixed during resampling.
- **Configuration:** nsb2 uses its published assets through Astropy's cache.
  Model versions, source URLs, input checksums and sample sizes are shown below.


In [ ]:
version_table = "| Package | Installed version |\n|---|---|\n"
version_table += "".join(f"| {name} | {version} |\n" for name, version in versions.items())
display(Markdown(version_table))
display(Markdown(
    f"**Inputs:** [Patat/FORS1]({PATAT_URLS[0]}) ([registered mirror]({PATAT_URLS[1]})); "
    f"[observed daily F10.7]({SOLAR_URL}).\n\n"
    f"**Registered SHA-256:**\n\n"
    f"- Patat archive: `{PATAT_SHA}`\n"
    f"- Solar source: `{SOLAR_SHA}`\n"
    f"- Selected cohort: `{selection['identity_sha256']}`\n\n"
    f"**Assessment:** {len(rows)} spectra / {len({r['night'] for r in rows})} nights; "
    f"365–650 nm, {UNIT}.\n\n"
    f"**Scenario status:** {availability}. **Rejected predictions:** {len(failures)}."
))
if failures:
    display(failures)
if not complete_campaign:
    raise RuntimeError("Incomplete scientific campaign: inspect availability and failures above")


### Historical results

Computed values and figures above come from this kernel. The following are
**historical context only**, never substituted for computation: matched relative
MAE was approximately continuum **35.027%**, Automatic/Oxygen **16.573%**, nsb2 **17.425%**,
and calibration baseline **15.245%**. Historical Oxygen−nsb2 relative-MAE difference
was −0.852 pp with paired 95% CI [−5.709, 5.576] pp: **superiority was not established**.
The baseline had lower relative MAE. Original benchmark revision
`1b2556727205694c7765ef0d254c5d1b301582ec`, Oxygen numerical revision
`168e73dae89060423eebc344871b0084d217c305`, and later merge
`57584ee298f25f8af7a6f169d71e87f8a59eea9a` are distinct identities.
A new version agreeing numerically is not a byte-identical historical replay.


### References and data credits

Sources to cite: [Patat (2008), A&A 481, 575](https://www.aanda.org/articles/aa/pdf/2008/14/aa9279-07.pdf);
[Roellinghoff, Spencer & Funk (2025), A&A 698, A212](https://www.aanda.org/articles/aa/pdf/2025/06/aa54532-25.pdf);
[nsb2 source and acknowledgements](https://gerritro.github.io/nsb2/acknowledgements.html);
[NSB Oxygen development evidence](https://github.com/VPRamon/NSB/blob/57584ee298f25f8af7a6f169d71e87f8a59eea9a/docs/nsb_components/airglow/validation/oxygen-and-calibration-v1.md).
The stellar path uses ESA Gaia data processed by DPAC and HEALPix/healpy; retain
original source credits and data terms. No private validation source is used.
